In [1]:
#!/usr/bin/env python3
"""
Ant Colony Optimization for the 15-Department Quadratic Assignment Problem (QAP)
Homework 7 — Intelligent Optimization, Spring 2026

Nugent et al. Nug15 benchmark — optimal solution = 1150 (symmetric flow*distance)
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time
import os
import json
from copy import deepcopy

# ==============================================================================
# 1. PROBLEM DATA — Nugent 15 QAP
# ==============================================================================
# Combined matrix: upper triangle = distances, lower triangle = flows, diagonal = 0
RAW_MATRIX = np.array([
    [ 0, 1, 2, 3, 4, 1, 2, 3, 4, 5, 2, 3, 4, 5, 6],
    [10, 0, 1, 2, 3, 2, 1, 2, 3, 4, 3, 2, 3, 4, 5],
    [ 0, 1, 0, 1, 2, 3, 2, 1, 2, 3, 4, 3, 2, 3, 4],
    [ 5, 3,10, 0, 1, 4, 3, 2, 1, 2, 5, 4, 3, 2, 3],
    [ 1, 2, 2, 1, 0, 5, 4, 3, 2, 1, 6, 5, 4, 3, 2],
    [ 0, 2, 0, 1, 3, 0, 1, 2, 3, 4, 1, 2, 3, 4, 5],
    [ 1, 2, 2, 5, 5, 2, 0, 1, 2, 3, 2, 1, 2, 3, 4],
    [ 2, 3, 5, 0, 5, 2, 6, 0, 1, 2, 3, 2, 1, 2, 3],
    [ 2, 2, 4, 0, 5, 1, 0, 5, 0, 1, 4, 3, 2, 1, 2],
    [ 2, 0, 5, 2, 1, 5, 1, 2, 0, 0, 5, 4, 3, 2, 1],
    [ 2, 2, 2, 1, 0, 0, 5,10,10, 0, 0, 1, 2, 3, 4],
    [ 0, 0, 2, 0, 3, 0, 5, 0, 5, 4, 5, 0, 1, 2, 3],
    [ 4,10, 5, 2, 0, 2, 5, 5,10, 0, 0, 3, 0, 1, 2],
    [ 0, 5, 5, 5, 5, 5, 1, 0, 0, 0, 5, 3,10, 0, 1],
    [ 0, 0, 5, 0, 5,10, 0, 0, 2, 5, 0, 0, 2, 4, 0],
])

N = 15  # number of departments / locations

# Extract flow matrix (lower triangle, make symmetric)
FLOW = np.zeros((N, N), dtype=int)
for i in range(N):
    for j in range(i):
        FLOW[i][j] = RAW_MATRIX[i][j]
        FLOW[j][i] = RAW_MATRIX[i][j]

# Extract distance matrix (upper triangle, make symmetric)
DIST = np.zeros((N, N), dtype=int)
for i in range(N):
    for j in range(i + 1, N):
        DIST[i][j] = RAW_MATRIX[i][j]
        DIST[j][i] = RAW_MATRIX[i][j]

OPTIMAL_COST = 1150  # known optimal for Nug15 (symmetric)


def evaluate(perm):
    """Evaluate the total flow cost for a given permutation.
    perm[i] = location assigned to department i.
    Cost = sum over all pairs (i,j) of flow[i][j] * distance[perm[i]][perm[j]]
    """
    perm = np.array(perm)
    d_sub = DIST[np.ix_(perm, perm)]
    return int(np.sum(FLOW * d_sub))


# ==============================================================================
# 2. ANT COLONY OPTIMIZATION
# ==============================================================================

def aco_qap(num_ants=10, max_iter=200, alpha=1.0, beta=2.0, rho=0.5,
            tau_init=1.0, q0=None, local_update=False, local_rho=0.1,
            elitist_fraction=None, seed=None):
    """
    Ant Colony Optimization for QAP.

    Encoding: perm[i] = location assigned to department i.
    Pheromone: tau[i][j] = desirability of assigning department i to location j.
    Local heuristic: eta[i][j] = 1 / (1 + sum of flow[i][k]*dist[j][loc_k])
                     for already placed departments k.

    Parameters:
    -----------
    num_ants : int - number of ants per iteration
    max_iter : int - maximum iterations
    alpha : float - pheromone importance (0 = heuristic only)
    beta : float - heuristic importance (0 = pheromone only)
    rho : float - evaporation rate (0 to 1)
    tau_init : float - initial pheromone value
    q0 : float or None - exploitation probability (ACS-style).
         If not None, with prob q0 choose best, else use roulette.
    local_update : bool - whether to apply local pheromone updating
    local_rho : float - local evaporation rate for local updating
    elitist_fraction : float or None - fraction of best ants that deposit pheromone
    seed : int - random seed

    Returns:
    --------
    best_perm, best_cost, history (best-so-far per iteration), iteration_best_history
    """
    rng = np.random.RandomState(seed)

    # Initialize pheromone matrix
    tau = np.full((N, N), tau_init, dtype=float)

    # Precompute the initial pheromone deposit value for local updating
    tau0 = tau_init

    best_perm = None
    best_cost = float('inf')
    history = []  # best-so-far at each iteration
    iter_best_history = []  # best of current iteration

    for iteration in range(max_iter):
        solutions = []
        costs = []

        for ant in range(num_ants):
            # Construct a solution: assign departments 0..14 to locations
            # We assign departments in a random order to add diversity
            dept_order = list(rng.permutation(N))
            perm = [-1] * N  # perm[dept] = location
            available_locs = list(range(N))

            for step_idx in range(N):
                dept = dept_order[step_idx]

                # Compute heuristic values for each available location
                probs = np.zeros(len(available_locs))

                for k, loc in enumerate(available_locs):
                    # Pheromone component
                    tau_val = tau[dept][loc] ** alpha if alpha > 0 else 1.0

                    # Heuristic: 1 / (1 + partial cost of placing dept at loc)
                    if beta > 0:
                        partial_cost = 0.0
                        for prev_dept in dept_order[:step_idx]:
                            prev_loc = perm[prev_dept]
                            partial_cost += FLOW[dept][prev_dept] * DIST[loc][prev_loc]
                        eta_val = (1.0 / (1.0 + partial_cost)) ** beta
                    else:
                        eta_val = 1.0

                    probs[k] = tau_val * eta_val

                # Normalize probabilities
                total = np.sum(probs)
                if total <= 0:
                    probs = np.ones(len(available_locs)) / len(available_locs)
                else:
                    probs = probs / total

                # q0 exploitation vs exploration
                if q0 is not None and rng.random() < q0:
                    # Exploitation: choose the best
                    chosen_idx = np.argmax(probs)
                else:
                    # Roulette wheel selection
                    chosen_idx = rng.choice(len(available_locs), p=probs)

                chosen_loc = available_locs[chosen_idx]
                perm[dept] = chosen_loc
                available_locs.remove(chosen_loc)

                # Local pheromone updating (reduce pheromone on used arc)
                if local_update:
                    tau[dept][chosen_loc] = (1 - local_rho) * tau[dept][chosen_loc] + local_rho * tau0

            cost = evaluate(perm)
            solutions.append(perm)
            costs.append(cost)

        # Find best of this iteration
        iter_best_idx = np.argmin(costs)
        iter_best_cost = costs[iter_best_idx]
        iter_best_perm = solutions[iter_best_idx]
        iter_best_history.append(iter_best_cost)

        # Update global best
        if iter_best_cost < best_cost:
            best_cost = iter_best_cost
            best_perm = list(iter_best_perm)

        history.append(best_cost)

        # === GLOBAL PHEROMONE UPDATE ===
        # Evaporation
        tau = (1 - rho) * tau

        # Deposit
        if elitist_fraction is not None:
            # Only top fraction of ants deposit
            num_elite = max(1, int(num_ants * elitist_fraction))
            sorted_indices = np.argsort(costs)[:num_elite]
            depositing_ants = sorted_indices
        else:
            # All ants deposit
            depositing_ants = range(num_ants)

        for ant_idx in depositing_ants:
            deposit = 1000.0 / costs[ant_idx]
            sol = solutions[ant_idx]
            for dept in range(N):
                tau[dept][sol[dept]] += deposit

    return best_perm, best_cost, history, iter_best_history


# ==============================================================================
# 3. RUN ALL 55 EXPERIMENTS
# ==============================================================================

SEEDS = [42, 123, 256, 789, 1024]
MAX_ITER = 200
results_all = []
run_id = 0
all_histories = {}

print("=" * 80)
print("ANT COLONY OPTIMIZATION FOR 15-DEPARTMENT QAP (Nugent Nug15)")
print(f"Known Optimal = {OPTIMAL_COST}")
print("=" * 80)

# ---- Base parameters ----
BASE_ANTS = 10
BASE_ALPHA = 1.0
BASE_BETA = 2.0
BASE_RHO = 0.5
BASE_TAU = 1.0

# ---------- E1: Base ACO — 5 runs ----------
print("\n" + "=" * 70)
print("E1: Base ACO — ants=10, alpha=1, beta=2, rho=0.5, tau0=1.0")
print("=" * 70)
for seed in SEEDS:
    run_id += 1
    t0 = time.perf_counter()
    best_perm, best_cost, hist, iter_hist = aco_qap(
        num_ants=BASE_ANTS, max_iter=MAX_ITER,
        alpha=BASE_ALPHA, beta=BASE_BETA, rho=BASE_RHO,
        tau_init=BASE_TAU, seed=seed
    )
    elapsed = time.perf_counter() - t0
    init_cost = iter_hist[0] if iter_hist else 0
    results_all.append({
        'run_id': run_id, 'experiment': 'E1_base', 'seed': seed,
        'ants': BASE_ANTS, 'alpha': BASE_ALPHA, 'beta': BASE_BETA,
        'rho': BASE_RHO, 'tau_init': BASE_TAU, 'variant': 'base',
        'best_cost': best_cost, 'best_perm': best_perm, 'runtime_s': elapsed
    })
    all_histories[run_id] = hist
    print(f"  Seed {seed:>5}: best={best_cost}, gap={best_cost - OPTIMAL_COST}, time={elapsed:.2f}s")

# ---------- E2: Population Size — 10 runs ----------
print("\n" + "=" * 70)
print("E2: Population Size — smaller (5 ants) and larger (20 ants)")
print("=" * 70)
for n_ants in [5, 20]:
    for seed in SEEDS:
        run_id += 1
        t0 = time.perf_counter()
        best_perm, best_cost, hist, iter_hist = aco_qap(
            num_ants=n_ants, max_iter=MAX_ITER,
            alpha=BASE_ALPHA, beta=BASE_BETA, rho=BASE_RHO,
            tau_init=BASE_TAU, seed=seed
        )
        elapsed = time.perf_counter() - t0
        results_all.append({
            'run_id': run_id, 'experiment': f'E2_pop_{n_ants}',
            'seed': seed, 'ants': n_ants, 'alpha': BASE_ALPHA,
            'beta': BASE_BETA, 'rho': BASE_RHO, 'tau_init': BASE_TAU,
            'variant': f'ants={n_ants}',
            'best_cost': best_cost, 'best_perm': best_perm, 'runtime_s': elapsed
        })
        all_histories[run_id] = hist
        print(f"  ants={n_ants}, Seed {seed:>5}: best={best_cost}, gap={best_cost - OPTIMAL_COST}, time={elapsed:.2f}s")

# Determine best population size
e2_results = [r for r in results_all if r['experiment'].startswith('E2_')]
e1_results = [r for r in results_all if r['experiment'] == 'E1_base']
pop_means = {}
for n_ants in [5, 10, 20]:
    if n_ants == 10:
        subset = e1_results
    else:
        subset = [r for r in e2_results if r['ants'] == n_ants]
    pop_means[n_ants] = np.mean([r['best_cost'] for r in subset])
BEST_ANTS = min(pop_means, key=pop_means.get)
print(f"\n  >> Best population size: {BEST_ANTS} (mean cost = {pop_means[BEST_ANTS]:.1f})")

# ---------- E3: Initial Pheromone — 10 runs ----------
print("\n" + "=" * 70)
print("E3: Initial Pheromone — smaller (0.1) and larger (5.0)")
print("=" * 70)
for tau_val in [0.1, 5.0]:
    for seed in SEEDS:
        run_id += 1
        t0 = time.perf_counter()
        best_perm, best_cost, hist, iter_hist = aco_qap(
            num_ants=BEST_ANTS, max_iter=MAX_ITER,
            alpha=BASE_ALPHA, beta=BASE_BETA, rho=BASE_RHO,
            tau_init=tau_val, seed=seed
        )
        elapsed = time.perf_counter() - t0
        results_all.append({
            'run_id': run_id, 'experiment': f'E3_tau_{tau_val}',
            'seed': seed, 'ants': BEST_ANTS, 'alpha': BASE_ALPHA,
            'beta': BASE_BETA, 'rho': BASE_RHO, 'tau_init': tau_val,
            'variant': f'tau0={tau_val}',
            'best_cost': best_cost, 'best_perm': best_perm, 'runtime_s': elapsed
        })
        all_histories[run_id] = hist
        print(f"  tau0={tau_val}, Seed {seed:>5}: best={best_cost}, gap={best_cost - OPTIMAL_COST}, time={elapsed:.2f}s")

# Determine best pheromone
tau_means = {}
for tau_val in [0.1, 1.0, 5.0]:
    if tau_val == 1.0:
        subset = [r for r in results_all if r['experiment'] == 'E1_base']
    else:
        subset = [r for r in results_all if r['experiment'] == f'E3_tau_{tau_val}']
    tau_means[tau_val] = np.mean([r['best_cost'] for r in subset])
BEST_TAU = min(tau_means, key=tau_means.get)
print(f"\n  >> Best initial pheromone: {BEST_TAU} (mean cost = {tau_means[BEST_TAU]:.1f})")

# ---------- E4: Evaporation Rate (rho) — 5 runs ----------
print("\n" + "=" * 70)
print("E4: Evaporation Rate — rho=0.2 and rho=0.8")
print("=" * 70)
for rho_val in [0.2, 0.8]:
    for seed in SEEDS:
        run_id += 1
        t0 = time.perf_counter()
        best_perm, best_cost, hist, iter_hist = aco_qap(
            num_ants=BEST_ANTS, max_iter=MAX_ITER,
            alpha=BASE_ALPHA, beta=BASE_BETA, rho=rho_val,
            tau_init=BEST_TAU, seed=seed
        )
        elapsed = time.perf_counter() - t0
        results_all.append({
            'run_id': run_id, 'experiment': f'E4_rho_{rho_val}',
            'seed': seed, 'ants': BEST_ANTS, 'alpha': BASE_ALPHA,
            'beta': BASE_BETA, 'rho': rho_val, 'tau_init': BEST_TAU,
            'variant': f'rho={rho_val}',
            'best_cost': best_cost, 'best_perm': best_perm, 'runtime_s': elapsed
        })
        all_histories[run_id] = hist
        print(f"  rho={rho_val}, Seed {seed:>5}: best={best_cost}, gap={best_cost - OPTIMAL_COST}, time={elapsed:.2f}s")

# Determine best rho
rho_means = {}
for rho_val in [0.2, 0.5, 0.8]:
    if rho_val == 0.5:
        # Use E1 base results with best ants and best tau
        subset = [r for r in results_all if r['experiment'] == 'E1_base']
    else:
        subset = [r for r in results_all if r['experiment'] == f'E4_rho_{rho_val}']
    rho_means[rho_val] = np.mean([r['best_cost'] for r in subset])
BEST_RHO = min(rho_means, key=rho_means.get)
print(f"\n  >> Best evaporation rate: {BEST_RHO} (mean cost = {rho_means[BEST_RHO]:.1f})")

# ---------- E5: Alpha = 0 (Heuristic Only) — 5 runs ----------
print("\n" + "=" * 70)
print(f"E5: Alpha=0 (heuristic only) — best combo: ants={BEST_ANTS}, tau={BEST_TAU}, rho={BEST_RHO}")
print("=" * 70)
for seed in SEEDS:
    run_id += 1
    t0 = time.perf_counter()
    best_perm, best_cost, hist, iter_hist = aco_qap(
        num_ants=BEST_ANTS, max_iter=MAX_ITER,
        alpha=0.0, beta=BASE_BETA, rho=BEST_RHO,
        tau_init=BEST_TAU, seed=seed
    )
    elapsed = time.perf_counter() - t0
    results_all.append({
        'run_id': run_id, 'experiment': 'E5_alpha0', 'seed': seed,
        'ants': BEST_ANTS, 'alpha': 0.0, 'beta': BASE_BETA,
        'rho': BEST_RHO, 'tau_init': BEST_TAU,
        'variant': 'alpha=0 (heuristic only)',
        'best_cost': best_cost, 'best_perm': best_perm, 'runtime_s': elapsed
    })
    all_histories[run_id] = hist
    print(f"  Seed {seed:>5}: best={best_cost}, gap={best_cost - OPTIMAL_COST}, time={elapsed:.2f}s")

# ---------- E6: Beta = 0 (Pheromone Only) — 5 runs ----------
print("\n" + "=" * 70)
print(f"E6: Beta=0 (pheromone only) — best combo: ants={BEST_ANTS}, tau={BEST_TAU}, rho={BEST_RHO}")
print("=" * 70)
for seed in SEEDS:
    run_id += 1
    t0 = time.perf_counter()
    best_perm, best_cost, hist, iter_hist = aco_qap(
        num_ants=BEST_ANTS, max_iter=MAX_ITER,
        alpha=BASE_ALPHA, beta=0.0, rho=BEST_RHO,
        tau_init=BEST_TAU, seed=seed
    )
    elapsed = time.perf_counter() - t0
    results_all.append({
        'run_id': run_id, 'experiment': 'E6_beta0', 'seed': seed,
        'ants': BEST_ANTS, 'alpha': BASE_ALPHA, 'beta': 0.0,
        'rho': BEST_RHO, 'tau_init': BEST_TAU,
        'variant': 'beta=0 (pheromone only)',
        'best_cost': best_cost, 'best_perm': best_perm, 'runtime_s': elapsed
    })
    all_histories[run_id] = hist
    print(f"  Seed {seed:>5}: best={best_cost}, gap={best_cost - OPTIMAL_COST}, time={elapsed:.2f}s")

# ---------- E7: q0 Exploitation — 5 runs ----------
print("\n" + "=" * 70)
print(f"E7: q0=0.5 exploitation — best combo: ants={BEST_ANTS}, tau={BEST_TAU}, rho={BEST_RHO}")
print("=" * 70)
for seed in SEEDS:
    run_id += 1
    t0 = time.perf_counter()
    best_perm, best_cost, hist, iter_hist = aco_qap(
        num_ants=BEST_ANTS, max_iter=MAX_ITER,
        alpha=BASE_ALPHA, beta=BASE_BETA, rho=BEST_RHO,
        tau_init=BEST_TAU, q0=0.5, seed=seed
    )
    elapsed = time.perf_counter() - t0
    results_all.append({
        'run_id': run_id, 'experiment': 'E7_q0', 'seed': seed,
        'ants': BEST_ANTS, 'alpha': BASE_ALPHA, 'beta': BASE_BETA,
        'rho': BEST_RHO, 'tau_init': BEST_TAU,
        'variant': 'q0=0.5',
        'best_cost': best_cost, 'best_perm': best_perm, 'runtime_s': elapsed
    })
    all_histories[run_id] = hist
    print(f"  Seed {seed:>5}: best={best_cost}, gap={best_cost - OPTIMAL_COST}, time={elapsed:.2f}s")

# ---------- E8: Local Pheromone Updating — 5 runs ----------
print("\n" + "=" * 70)
print(f"E8: Local pheromone updating — best combo: ants={BEST_ANTS}, tau={BEST_TAU}, rho={BEST_RHO}")
print("=" * 70)
for seed in SEEDS:
    run_id += 1
    t0 = time.perf_counter()
    best_perm, best_cost, hist, iter_hist = aco_qap(
        num_ants=BEST_ANTS, max_iter=MAX_ITER,
        alpha=BASE_ALPHA, beta=BASE_BETA, rho=BEST_RHO,
        tau_init=BEST_TAU, local_update=True, local_rho=0.1, seed=seed
    )
    elapsed = time.perf_counter() - t0
    results_all.append({
        'run_id': run_id, 'experiment': 'E8_local', 'seed': seed,
        'ants': BEST_ANTS, 'alpha': BASE_ALPHA, 'beta': BASE_BETA,
        'rho': BEST_RHO, 'tau_init': BEST_TAU,
        'variant': 'local update (rho_l=0.1)',
        'best_cost': best_cost, 'best_perm': best_perm, 'runtime_s': elapsed
    })
    all_histories[run_id] = hist
    print(f"  Seed {seed:>5}: best={best_cost}, gap={best_cost - OPTIMAL_COST}, time={elapsed:.2f}s")

# ---------- E9: Elitist Pheromone Depositing — 5 runs ----------
print("\n" + "=" * 70)
print(f"E9: Elitist (top 30%) pheromone + local updating — best combo")
print("=" * 70)
for seed in SEEDS:
    run_id += 1
    t0 = time.perf_counter()
    best_perm, best_cost, hist, iter_hist = aco_qap(
        num_ants=BEST_ANTS, max_iter=MAX_ITER,
        alpha=BASE_ALPHA, beta=BASE_BETA, rho=BEST_RHO,
        tau_init=BEST_TAU, local_update=True, local_rho=0.1,
        elitist_fraction=0.3, seed=seed
    )
    elapsed = time.perf_counter() - t0
    results_all.append({
        'run_id': run_id, 'experiment': 'E9_elitist', 'seed': seed,
        'ants': BEST_ANTS, 'alpha': BASE_ALPHA, 'beta': BASE_BETA,
        'rho': BEST_RHO, 'tau_init': BEST_TAU,
        'variant': 'elitist 30% + local update',
        'best_cost': best_cost, 'best_perm': best_perm, 'runtime_s': elapsed
    })
    all_histories[run_id] = hist
    print(f"  Seed {seed:>5}: best={best_cost}, gap={best_cost - OPTIMAL_COST}, time={elapsed:.2f}s")



ANT COLONY OPTIMIZATION FOR 15-DEPARTMENT QAP (Nugent Nug15)
Known Optimal = 1150

E1: Base ACO — ants=10, alpha=1, beta=2, rho=0.5, tau0=1.0
  Seed    42: best=1292, gap=142, time=0.59s
  Seed   123: best=1282, gap=132, time=0.56s
  Seed   256: best=1282, gap=132, time=0.55s
  Seed   789: best=1260, gap=110, time=0.56s
  Seed  1024: best=1290, gap=140, time=0.56s

E2: Population Size — smaller (5 ants) and larger (20 ants)
  ants=5, Seed    42: best=1318, gap=168, time=0.29s
  ants=5, Seed   123: best=1308, gap=158, time=0.28s
  ants=5, Seed   256: best=1304, gap=154, time=0.29s
  ants=5, Seed   789: best=1272, gap=122, time=0.28s
  ants=5, Seed  1024: best=1306, gap=156, time=0.29s
  ants=20, Seed    42: best=1294, gap=144, time=1.14s
  ants=20, Seed   123: best=1278, gap=128, time=1.13s
  ants=20, Seed   256: best=1258, gap=108, time=1.11s
  ants=20, Seed   789: best=1266, gap=116, time=1.12s
  ants=20, Seed  1024: best=1274, gap=124, time=1.11s

  >> Best population size: 20 (mean 

In [2]:
# ==============================================================================
# 4. SAVE RESULTS
# ==============================================================================

results_df = pd.DataFrame(results_all)
os.makedirs('ACO_Results', exist_ok=True)

save_cols = [c for c in results_df.columns if c != 'best_perm']
results_df[save_cols].to_csv('ACO_Results/all_55_runs.csv', index=False)

results_df['best_perm_str'] = results_df['best_perm'].apply(str)
results_df.drop(columns=['best_perm']).to_csv('ACO_Results/all_55_runs_full.csv', index=False)

with open('ACO_Results/histories.json', 'w') as f:
    json.dump({str(k): v for k, v in all_histories.items()}, f)

print(f"\n{'='*80}")
print(f"ALL {run_id} RUNS COMPLETE")
print(f"{'='*80}")
print(f"Overall Best Cost: {results_df['best_cost'].min()}")
print(f"Optimal: {OPTIMAL_COST}")
print(f"Mean Cost: {results_df['best_cost'].mean():.1f}")
print(f"Best parameters chosen: ants={BEST_ANTS}, tau={BEST_TAU}, rho={BEST_RHO}")


ALL 60 RUNS COMPLETE
Overall Best Cost: 1196
Optimal: 1150
Mean Cost: 1275.3
Best parameters chosen: ants=20, tau=5.0, rho=0.8


In [16]:
# ==============================================================================
# 5. ANALYSIS & PLOTS
# ==============================================================================

COLORS = {
    'E1_base': '#2E86AB',
    'E2_pop_5': '#A23B72',
    'E2_pop_20': '#F18F01',
    'E3_tau_0.1': '#C73E1D',
    'E3_tau_5.0': '#3B1F2B',
    'E4_rho_0.2': '#44BBA4',
    'E4_rho_0.8': '#E94F37',
    'E5_alpha0': '#8B5CF6',
    'E6_beta0': '#059669',
    'E7_q0': '#D97706',
    'E8_local': '#DC2626',
    'E9_elitist': '#2563EB',
}

EXP_LABELS = {
    'E1_base': 'E1: Base ACO',
    'E2_pop_5': 'E2: 5 Ants',
    'E2_pop_20': 'E2: 20 Ants',
    'E3_tau_0.1': 'E3: tau=0.1',
    'E3_tau_5.0': 'E3: tau=5.0',
    'E4_rho_0.2': 'E4: rho=0.2',
    'E4_rho_0.8': 'E4: rho=0.8',
    'E5_alpha0': 'E5: alpha=0',
    'E6_beta0': 'E6: beta=0',
    'E7_q0': 'E7: q0=0.5',
    'E8_local': 'E8: Local Update',
    'E9_elitist': 'E9: Elitist',
}

plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 12,
    'figure.dpi': 150, 'savefig.dpi': 150, 'savefig.bbox': 'tight'
})

# --- Fig 1: Bar chart of near-optimums by seed for E1 (base) ---
fig, ax = plt.subplots(figsize=(8, 5))
e1 = results_df[results_df['experiment'] == 'E1_base']
e1_seeds = e1['seed'].values
e1_costs = e1['best_cost'].values
bars = ax.bar(range(len(e1_seeds)), e1_costs, color=COLORS['E1_base'], alpha=0.8, edgecolor='black')
ax.axhline(OPTIMAL_COST, color='red', linestyle='--', linewidth=1.5, label=f'Optimal ({OPTIMAL_COST})')
ax.axhline(np.mean(e1_costs), color='orange', linestyle='-', linewidth=1.5, label=f'Mean ({np.mean(e1_costs):.0f})')
ax.set_xticks(range(len(e1_seeds)))
ax.set_xticklabels([f'Seed {s}' for s in e1_seeds])
ax.set_ylabel('Best Cost')
ax.set_yscale('log')
ax.legend()
plt.tight_layout()
plt.savefig('ACO_Results/fig1_bar_base_seeds.png', dpi=300)
plt.close()
print("Saved: fig1_bar_base_seeds.png")

# --- Fig 2: Convergence curves for E1 ---
fig, ax = plt.subplots(figsize=(10, 6))
for _, row in e1.iterrows():
    hist = all_histories[row['run_id']]
    ax.plot(hist, linewidth=1.5, label=f"Seed {row['seed']}")
ax.axhline(OPTIMAL_COST, color='red', linestyle='--', linewidth=1.2, label=f'Optimal ({OPTIMAL_COST})')
ax.set_xlabel('Iteration')
ax.set_ylabel('Best Cost So Far')
ax.legend()
plt.tight_layout()
plt.savefig('ACO_Results/fig2_convergence_base.png', dpi=300)
plt.close()
print("Saved: fig2_convergence_base.png")

# --- Fig 3: Boxplot for population size comparison (E1, E2) ---
fig, ax = plt.subplots(figsize=(8, 5))
pop_exps = ['E2_pop_5', 'E1_base', 'E2_pop_20']
pop_labels = ['5 Ants', '10 Ants\n(Base)', '20 Ants']
data_pop = [results_df[results_df['experiment'] == e]['best_cost'].values for e in pop_exps]
bp = ax.boxplot(data_pop, labels=pop_labels, patch_artist=True, widths=0.5)
pop_colors = [COLORS['E2_pop_5'], COLORS['E1_base'], COLORS['E2_pop_20']]
for patch, c in zip(bp['boxes'], pop_colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax.axhline(OPTIMAL_COST, color='red', linestyle='--', linewidth=1.5, label=f'Optimal ({OPTIMAL_COST})')
ax.set_ylabel('Best Cost Found')
#ax.set_yscale('log')
ax.legend()
plt.tight_layout()
plt.savefig('ACO_Results/fig3_boxplot_population.png', dpi=300)
plt.close()
print("Saved: fig3_boxplot_population.png")

# --- Fig 4: Boxplot for initial pheromone (E1, E3) ---
fig, ax = plt.subplots(figsize=(8, 5))
tau_exps = ['E3_tau_0.1', 'E1_base', 'E3_tau_5.0']
tau_labels = ['tau=0.1', 'tau=1.0\n(Base)', 'tau=5.0']
data_tau = [results_df[results_df['experiment'] == e]['best_cost'].values for e in tau_exps]
bp2 = ax.boxplot(data_tau, labels=tau_labels, patch_artist=True, widths=0.5)
tau_colors_list = [COLORS['E3_tau_0.1'], COLORS['E1_base'], COLORS['E3_tau_5.0']]
for patch, c in zip(bp2['boxes'], tau_colors_list):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax.axhline(OPTIMAL_COST, color='red', linestyle='--', linewidth=1.5, label=f'Optimal ({OPTIMAL_COST})')
ax.set_ylabel('Best Cost Found')
ax.legend()
plt.tight_layout()
plt.savefig('ACO_Results/fig4_boxplot_pheromone.png')
plt.close()
print("Saved: fig4_boxplot_pheromone.png")

# --- Fig 5: Boxplot for evaporation rate (E1/best, E4) ---
fig, ax = plt.subplots(figsize=(8, 5))
rho_exps = ['E4_rho_0.2', 'E1_base', 'E4_rho_0.8']
rho_labels = ['rho=0.2', 'rho=0.5\n(Base)', 'rho=0.8']
data_rho = [results_df[results_df['experiment'] == e]['best_cost'].values for e in rho_exps]
bp3 = ax.boxplot(data_rho, labels=rho_labels, patch_artist=True, widths=0.5)
rho_colors = [COLORS['E4_rho_0.2'], COLORS['E1_base'], COLORS['E4_rho_0.8']]
for patch, c in zip(bp3['boxes'], rho_colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax.axhline(OPTIMAL_COST, color='red', linestyle='--', linewidth=1.5, label=f'Optimal ({OPTIMAL_COST})')
ax.set_ylabel('Best Cost Found')
ax.legend()
plt.tight_layout()
plt.savefig('ACO_Results/fig5_boxplot_rho.png')
plt.close()
print("Saved: fig5_boxplot_rho.png")

# --- Fig 6: Boxplot for alpha=0 vs beta=0 vs full model ---
fig, ax = plt.subplots(figsize=(6, 4))
ab_exps = ['E1_base', 'E5_alpha0', 'E6_beta0']
ab_labels = ['Full Model\n(alpha=1, beta=2)', 'Heuristic Only\n(alpha=0)', 'Pheromone Only\n(beta=0)']
data_ab = [results_df[results_df['experiment'] == e]['best_cost'].values for e in ab_exps]
bp4 = ax.boxplot(data_ab, labels=ab_labels, patch_artist=True, widths=0.5)
ab_colors = [COLORS['E1_base'], COLORS['E5_alpha0'], COLORS['E6_beta0']]
for patch, c in zip(bp4['boxes'], ab_colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax.axhline(OPTIMAL_COST, color='red', linestyle='--', linewidth=1.5, label=f'Optimal ({OPTIMAL_COST})')
ax.set_ylabel('Best Cost Found')
ax.legend()
plt.tight_layout()
plt.savefig('ACO_Results/fig6_boxplot_alpha_beta.png', dpi=300)
plt.close()
print("Saved: fig6_boxplot_alpha_beta.png")

# --- Fig 7: Convergence comparison alpha=0 vs beta=0 vs full (one seed) ---
fig, ax = plt.subplots(figsize=(8, 5))
for exp_name in ['E1_base', 'E5_alpha0', 'E6_beta0']:
    row = results_df[(results_df['experiment'] == exp_name) & (results_df['seed'] == SEEDS[0])].iloc[0]
    hist = all_histories[row['run_id']]
    ax.plot(hist, linewidth=1.5, label=EXP_LABELS.get(exp_name, exp_name), color=COLORS[exp_name])
ax.axhline(OPTIMAL_COST, color='red', linestyle='--', linewidth=1.2, label=f'Optimal ({OPTIMAL_COST})')
ax.set_xlabel('Iteration')
ax.set_ylabel('Best Cost So Far')
ax.legend()
plt.tight_layout()
plt.savefig('ACO_Results/fig7_convergence_alpha_beta.png', dpi=300)
plt.close()
print("Saved: fig7_convergence_alpha_beta.png")

# --- Fig 8: Boxplot for q0 vs base ---
fig, ax = plt.subplots(figsize=(6, 4))
q0_exps = ['E1_base', 'E7_q0']
q0_labels = ['No q0\n(Base)', 'q0=0.5']
data_q0 = [results_df[results_df['experiment'] == e]['best_cost'].values for e in q0_exps]
bp5 = ax.boxplot(data_q0, labels=q0_labels, patch_artist=True, widths=0.4)
q0_colors = [COLORS['E1_base'], COLORS['E7_q0']]
for patch, c in zip(bp5['boxes'], q0_colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax.axhline(OPTIMAL_COST, color='red', linestyle='--', linewidth=1.5, label=f'Optimal ({OPTIMAL_COST})')
ax.set_ylabel('Best Cost Found')
ax.legend()
plt.tight_layout()
plt.savefig('ACO_Results/fig8_boxplot_q0.png', dpi=300)
plt.close()
print("Saved: fig8_boxplot_q0.png")

# --- Fig 9: Boxplot for local update + elitist ---
fig, ax = plt.subplots(figsize=(7, 4))
le_exps = ['E1_base', 'E8_local', 'E9_elitist']
le_labels = ['Base ACO', 'Local Update', 'Elitist +\nLocal Update']
data_le = [results_df[results_df['experiment'] == e]['best_cost'].values for e in le_exps]
bp6 = ax.boxplot(data_le, labels=le_labels, patch_artist=True, widths=0.5)
le_colors = [COLORS['E1_base'], COLORS['E8_local'], COLORS['E9_elitist']]
for patch, c in zip(bp6['boxes'], le_colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax.axhline(OPTIMAL_COST, color='red', linestyle='--', linewidth=1.5, label=f'Optimal ({OPTIMAL_COST})')
ax.set_ylabel('Best Cost Found')
ax.legend()
plt.tight_layout()
plt.savefig('ACO_Results/fig9_boxplot_local_elitist.png', dpi=400)
plt.close()
print("Saved: fig9_boxplot_local_elitist.png")

# --- Fig 10: Overall boxplot all experiments ---
fig, ax = plt.subplots(figsize=(10, 5))
all_exps_unique = results_df['experiment'].unique()
data_all = [results_df[results_df['experiment'] == e]['best_cost'].values for e in all_exps_unique]
labels_all = [EXP_LABELS.get(e, e) for e in all_exps_unique]
bp7 = ax.boxplot(data_all, labels=labels_all, patch_artist=True, widths=0.6)
for patch, exp in zip(bp7['boxes'], all_exps_unique):
    patch.set_facecolor(COLORS.get(exp, '#AAAAAA'))
    patch.set_alpha(0.7)
ax.axhline(OPTIMAL_COST, color='red', linestyle='--', linewidth=1.5, label=f'Optimal ({OPTIMAL_COST})')
ax.set_ylabel('Best Cost Found')
ax.legend(loc='upper right')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('ACO_Results/fig10_boxplot_all.png', dpi=400)
plt.close()
print("Saved: fig10_boxplot_all.png")

# --- Fig 11: Summary mean ± std bar chart ---
fig, ax = plt.subplots(figsize=(10, 5))
means = [results_df[results_df['experiment'] == e]['best_cost'].mean() for e in all_exps_unique]
stds = [results_df[results_df['experiment'] == e]['best_cost'].std() for e in all_exps_unique]
bars = ax.bar(range(len(all_exps_unique)), means, yerr=stds, capsize=6,
              color=[COLORS.get(e, '#AAAAAA') for e in all_exps_unique], edgecolor='black', alpha=0.85)
ax.axhline(OPTIMAL_COST, color='red', linestyle='--', linewidth=1.5, label=f'Optimal ({OPTIMAL_COST})')
ax.set_xticks(range(len(all_exps_unique)))
ax.set_xticklabels([EXP_LABELS.get(e, e) for e in all_exps_unique], rotation=30, ha='right')
ax.set_ylabel('Best Cost (Mean +/- Std)')
ax.legend()
y_min = max(OPTIMAL_COST - 20, min(m - s for m, s in zip(means, stds)) - 20)
y_max = max(m + s for m, s in zip(means, stds)) + 30
ax.set_ylim(y_min, y_max)
plt.tight_layout()
plt.savefig('ACO_Results/fig11_mean_std_bar.png', dpi=400)
plt.close()
print("Saved: fig11_mean_std_bar.png")

# --- Fig 12: Convergence comparison all key experiments (one seed) ---
fig, ax = plt.subplots(figsize=(10, 5))
key_exps = ['E1_base', 'E5_alpha0', 'E6_beta0', 'E7_q0', 'E8_local', 'E9_elitist']
for exp_name in key_exps:
    row = results_df[(results_df['experiment'] == exp_name) & (results_df['seed'] == SEEDS[0])]
    if len(row) > 0:
        row = row.iloc[0]
        hist = all_histories[row['run_id']]
        ax.plot(hist, linewidth=1.5, label=EXP_LABELS.get(exp_name, exp_name), color=COLORS[exp_name])
ax.axhline(OPTIMAL_COST, color='red', linestyle='--', linewidth=1.2, label=f'Optimal ({OPTIMAL_COST})')
ax.set_xlabel('Iteration')
ax.set_ylabel('Best Cost So Far')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('ACO_Results/fig12_convergence_all.png', dpi=400)
plt.close()
print("Saved: fig12_convergence_all.png")

# --- Fig 13: Heatmap of best cost by experiment x seed ---
fig, ax = plt.subplots(figsize=(12, 7))
pivot = results_df.pivot_table(index='experiment', columns='seed', values='best_cost')
pivot = pivot.reindex(all_exps_unique)
pivot.index = [EXP_LABELS.get(e, e) for e in all_exps_unique]
vmin = OPTIMAL_COST
vmax = pivot.values.max()
im = ax.imshow(pivot.values, cmap='RdYlGn_r', aspect='auto', vmin=vmin, vmax=vmax)
ax.set_xticks(range(len(SEEDS)))
ax.set_xticklabels([f'Seed {s}' for s in SEEDS])
ax.set_yticks(range(len(all_exps_unique)))
ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = int(pivot.values[i, j])
        color = 'white' if val > (vmin + vmax) / 2 else 'black'
        fontweight = 'bold' if val == OPTIMAL_COST else 'normal'
        ax.text(j, i, str(val), ha='center', va='center', fontsize=10,
                color=color, fontweight=fontweight)
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Best Cost')
plt.tight_layout()
#plt.grid ('both', linestyle='-')
plt.savefig('ACO_Results/fig13_heatmap.png', dpi=400)
plt.close()
print("Saved: fig13_heatmap.png")

# --- Fig 14: Best solution grid layout ---
import re
best_row = results_df.loc[results_df['best_cost'].idxmin()]
best_perm_val = best_row['best_perm'] if 'best_perm' in results_df.columns else None
if best_perm_val is None or not isinstance(best_perm_val, list):
    nums = list(range(15))  # fallback
else:
    nums = best_perm_val

fig, ax = plt.subplots(figsize=(7, 4))
grid = np.zeros((3, 5), dtype=int)
for dept in range(15):
    loc = nums[dept]
    grid[loc // 5][loc % 5] = dept + 1
for r in range(3):
    for c in range(5):
        color = '#D5E8F0' if (r + c) % 2 == 0 else '#E8F4E8'
        rect = plt.Rectangle((c, 2 - r), 1, 1, facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        ax.text(c + 0.5, 2 - r + 0.5, f'Dept {grid[r][c]}',
                ha='center', va='center', fontsize=12, fontweight='bold')
ax.set_xlim(0, 5); ax.set_ylim(0, 3); ax.set_aspect('equal')
ax.set_xticks([0.5, 1.5, 2.5, 3.5, 4.5])
ax.set_xticklabels(['Col 1', 'Col 2', 'Col 3', 'Col 4', 'Col 5'])
ax.set_yticks([0.5, 1.5, 2.5])
ax.set_yticklabels(['Row 3', 'Row 2', 'Row 1'])
plt.tight_layout()
plt.savefig('ACO_Results/fig14_grid_layout.png', dpi=400)
plt.close()
print("Saved: fig14_grid_layout.png")

# --- Summary Table ---
summary_data = []
for exp in all_exps_unique:
    sub = results_df[results_df['experiment'] == exp]
    summary_data.append({
        'Experiment': EXP_LABELS.get(exp, exp),
        'Runs': len(sub),
        'Mean': f"{sub['best_cost'].mean():.1f}",
        'Std': f"{sub['best_cost'].std():.1f}",
        'Min': f"{sub['best_cost'].min()}",
        'Max': f"{sub['best_cost'].max()}",
        'Avg Runtime': f"{sub['runtime_s'].mean():.2f}s"
    })
summary_df = pd.DataFrame(summary_data)
summary_df.to_csv('ACO_Results/summary_table.csv', index=False)

print("\n" + "=" * 80)
print("SUMMARY TABLE")
print("=" * 80)
print(summary_df.to_string(index=False))

best_overall = results_df.loc[results_df['best_cost'].idxmin()]
print(f"\n{'='*80}")
print("BEST SOLUTION FOUND ACROSS ALL RUNS")
print(f"{'='*80}")
print(f"  Experiment:   {best_overall['experiment']}")
print(f"  Seed:         {best_overall['seed']}")
print(f"  Best Cost:    {best_overall['best_cost']}")
print(f"  Optimal:      {OPTIMAL_COST}")
print(f"  Gap:          {best_overall['best_cost'] - OPTIMAL_COST}")
print(f"  Runtime:      {best_overall['runtime_s']:.2f}s")

print("\nAll plots and data saved to ACO_Results/")
print("DONE.")


Saved: fig1_bar_base_seeds.png
Saved: fig2_convergence_base.png
Saved: fig3_boxplot_population.png
Saved: fig4_boxplot_pheromone.png


/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_65306/244067895.py:77: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_pop, labels=pop_labels, patch_artist=True, widths=0.5)
/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_65306/244067895.py:96: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp2 = ax.boxplot(data_tau, labels=tau_labels, patch_artist=True, widths=0.5)
/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_65306/244067895.py:114: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp3 = ax.boxplot(data_rho, labels=rho_labels, patch_artist=True, widths=0.5)
/var/folde

Saved: fig5_boxplot_rho.png
Saved: fig6_boxplot_alpha_beta.png
Saved: fig7_convergence_alpha_beta.png
Saved: fig8_boxplot_q0.png


/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_65306/244067895.py:165: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp5 = ax.boxplot(data_q0, labels=q0_labels, patch_artist=True, widths=0.4)
/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_65306/244067895.py:183: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp6 = ax.boxplot(data_le, labels=le_labels, patch_artist=True, widths=0.5)
/var/folders/w2/8rhf6prd1mv2_x48l625r7nw0000gn/T/ipykernel_65306/244067895.py:201: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp7 = ax.boxplot(data_all, labels=labels_all, patch_artist=True, widths=0.6)


Saved: fig9_boxplot_local_elitist.png
Saved: fig10_boxplot_all.png
Saved: fig11_mean_std_bar.png
Saved: fig12_convergence_all.png
Saved: fig13_heatmap.png
Saved: fig14_grid_layout.png

SUMMARY TABLE
      Experiment  Runs   Mean  Std  Min  Max Avg Runtime
    E1: Base ACO     5 1281.2 12.7 1260 1292       0.57s
      E2: 5 Ants     5 1301.6 17.4 1272 1318       0.29s
     E2: 20 Ants     5 1274.0 13.6 1258 1294       1.12s
     E3: tau=0.1     5 1282.8  8.7 1272 1296       1.13s
     E3: tau=5.0     5 1266.0  7.2 1256 1276       1.13s
     E4: rho=0.2     5 1263.2 15.4 1244 1280       1.13s
     E4: rho=0.8     5 1260.4 19.2 1236 1288       1.15s
     E5: alpha=0     5 1280.0 20.5 1256 1304       1.10s
      E6: beta=0     5 1312.0 32.3 1286 1352       0.40s
      E7: q0=0.5     5 1302.4 40.8 1236 1344       1.05s
E8: Local Update     5 1270.8 24.7 1230 1292       1.14s
     E9: Elitist     5 1209.6 15.3 1196 1226       1.14s

BEST SOLUTION FOUND ACROSS ALL RUNS
  Experiment:   E9_elit